In [1]:
# Install Packages
!pip install scanpy anndata scikit-misc torch_geometric
# ------------------------ Load Packages ---------------------------------------
import numpy as np
import scanpy as sc
import scipy
import sklearn
import anndata
from typing import Optional
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from scipy.spatial.distance import cdist
import torch
from torch_geometric.data import Data



# ------------------------ Process ---------------------------------------



def clr_normalize_each_cell(adata, inplace=True):
    """
    Perform CLR (Centered Log-Ratio) normalization on protein (ADT) data per cell.

    Args:
        adata (AnnData): The input AnnData object containing the expression matrix.
        inplace (bool): Whether to modify the original AnnData object or return a copy.

    Returns:
        AnnData: The CLR-normalized AnnData object.
    """
    def seurat_clr(x):
        s = np.sum(np.log1p(x[x > 0]))
        exp = np.exp(s / len(x))
        return np.log1p(x / exp)

    if not inplace:
        adata = adata.copy()

    adata.X = np.apply_along_axis(
        seurat_clr, 1, (adata.X.toarray() if scipy.sparse.issparse(adata.X) else np.array(adata.X))
    )
    return adata



def pca(adata, use_reps=None, n_comps=10):
    """
    Perform PCA (Principal Component Analysis) on the input AnnData object.

    Args:
        adata (AnnData): The input data.
        use_reps (str, optional): Use a precomputed feature representation (e.g., adata.obsm['X_pca']).
        n_comps (int): Number of principal components.

    Returns:
        np.ndarray: The PCA-transformed matrix.
    """
    from sklearn.decomposition import PCA
    pca_model = PCA(n_components=n_comps)

    if use_reps is not None:
        feat_pca = pca_model.fit_transform(adata.obsm[use_reps])
    else:
        feat_pca = pca_model.fit_transform(adata.X.toarray() if scipy.sparse.issparse(adata.X) else adata.X)

    return feat_pca


def normalize(adata, highly_genes=3000):
    """
    Normalize RNA data and extract highly variable genes.

    Args:
        adata (AnnData): The input RNA data.
        highly_genes (int): Number of top highly variable genes to retain.

    Returns:
        AnnData: The processed AnnData object.
    """
    sc.pp.filter_genes(adata, min_cells=100)
    sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=highly_genes)
    adata = adata[:, adata.var['highly_variable']].copy()
    adata.X = adata.X / np.sum(adata.X, axis=1).reshape(-1, 1) * 10000
    sc.pp.scale(adata, zero_center=False, max_value=10)
    return adata



def Protein(adata):
    """
    Normalize ADT (Protein) data using CLR and scale it.

    Args:
        adata (AnnData): The input ADT data.

    Returns:
        AnnData: The processed ADT data.
    """
    adata = clr_normalize_each_cell(adata)
    sc.pp.scale(adata)
    return adata


# def tfidf(X):
#     """
#     Compute TF-IDF matrix for input count matrix.

#     Args:
#         X (np.ndarray or sparse): Count matrix.

#     Returns:
#         np.ndarray or sparse: TF-IDF normalized matrix.
#     """
#     idf = X.shape[0] / X.sum(axis=0)
#     if scipy.sparse.issparse(X):
#         tf = X.multiply(1 / X.sum(axis=1))
#         return tf.multiply(idf)
#     else:
#         tf = X / X.sum(axis=1, keepdims=True)
#         return tf * idf

def tfidf(X):
    """
    Compute TF-IDF matrix for input count matrix.

    Args:
        X (np.ndarray or sparse): Count matrix.

    Returns:
        np.ndarray or sparse: TF-IDF normalized matrix in CSR sparse format.
    """
    import scipy.sparse as sp
    idf = X.shape[0] / X.sum(axis=0)
    if sp.issparse(X):
        tf = X.multiply(1 / X.sum(axis=1))
        return sp.csr_matrix(tf.multiply(idf))
    else:
        tf = X / X.sum(axis=1, keepdims=True)
        return tf * idf




def lsi(adata: anndata.AnnData, n_components, use_highly_variable: Optional[bool] = None, **kwargs) -> None:
    """
    Perform LSI (Latent Semantic Indexing) on AnnData.

    Args:
        adata (AnnData): The input data.
        n_components (int): Number of components.
        use_highly_variable (bool): Whether to use only highly variable genes.
    """
    if use_highly_variable is None:
        use_highly_variable = "highly_variable" in adata.var
    adata_use = adata[:, adata.var["highly_variable"]] if use_highly_variable else adata
    X = tfidf(adata_use.X)
    X_norm = sklearn.preprocessing.Normalizer(norm="l1").fit_transform(X)
    X_norm = np.log1p(X_norm * 1e4)
    X_lsi = sklearn.utils.extmath.randomized_svd(X_norm, n_components, **kwargs)[0]
    X_lsi -= X_lsi.mean(axis=1, keepdims=True)
    X_lsi /= X_lsi.std(axis=1, ddof=1, keepdims=True)
    adata.obsm["X_lsi"] = X_lsi[:, 1:]



def adata_preprocess_1(adata, min_cells=100, pca_n_comps=2000, HVG=3000):
    """
    Basic preprocessing pipeline for RNA modality.

    Args:
        adata (AnnData): RNA expression data.
        min_cells (int): Minimum cells per gene.
        pca_n_comps (int): Not used in current function.
        HVG (int): Number of highly variable genes.

    Returns:
        np.ndarray: Processed RNA expression matrix.
    """
    sc.pp.filter_genes(adata, min_cells=min_cells)
    sc.pp.filter_cells(adata, min_counts=3)
    sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=HVG)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.scale(adata, zero_center=False, max_value=10)
    return adata[:, adata.var['highly_variable']].X




class DualGraphData(Data):
    """
    Custom PyTorch Geometric Data structure for dual-graph learning.

    Attributes:
        x_RNA: RNA feature matrix.
        x_ADT: ADT feature matrix.
        sim_edge_index: Indices for similarity-based edges.
        sim_edge_weight: Weights for similarity edges.
        dist_edge_index: Indices for spatial distance-based edges.
        dist_edge_weight: Weights for spatial edges.
        common_edge_index: Overlap of similarity and spatial edges.
        common_edge_weight: Weights for common edges.
    """
    def __init__(self, x_RNA, x_ADT, sim_edge_index, sim_edge_weight,
                 dist_edge_index, dist_edge_weight, common_edge_index, common_edge_weight):
        super().__init__()
        self.x_RNA = x_RNA
        self.x_ADT = x_ADT
        self.sim_edge_index = sim_edge_index
        self.sim_edge_weight = sim_edge_weight
        self.dist_edge_index = dist_edge_index
        self.dist_edge_weight = dist_edge_weight
        self.common_edge_index = common_edge_index
        self.common_edge_weight = common_edge_weight



def build_dual_graph(RNA_expression, ADT_expression, cell_positions, device='cpu', num_neighbors=15):
    """
    Build a dual graph based on expression similarity and spatial proximity.

    Args:
        RNA_expression (np.ndarray): RNA expression matrix.
        ADT_expression (np.ndarray): ADT expression matrix.
        cell_positions (np.ndarray): 2D coordinates of cells.
        device (str): Device to place tensors ('cpu' or 'cuda').
        num_neighbors (int): Number of neighbors for KNN.

    Returns:
        DualGraphData: PyTorch Geometric-compatible graph object.
    """
    similarity_matrix = cosine_similarity(RNA_expression)

    nbrs = NearestNeighbors(n_neighbors=num_neighbors + 1, metric='cosine').fit(RNA_expression)
    distances, indices = nbrs.kneighbors(RNA_expression)

    adjacency_matrix = np.zeros_like(similarity_matrix, dtype=int)
    for i in range(len(RNA_expression)):
        for j in indices[i][1:]:
            adjacency_matrix[i, j] = 1
            adjacency_matrix[j, i] = 1

    sim_edge_index = torch.tensor(np.array(np.nonzero(adjacency_matrix)), dtype=torch.long).to(device)
    sim_edge_weight = torch.tensor(similarity_matrix[adjacency_matrix > 0], dtype=torch.float).to(device)

    distance_matrix = cdist(cell_positions, cell_positions, metric='euclidean')
    knn_graph = kneighbors_graph(cell_positions, n_neighbors=num_neighbors, mode='distance', include_self=False)
    knn_graph = knn_graph.maximum(knn_graph.T)

    dist_edge_index = torch.tensor(knn_graph.nonzero(), dtype=torch.long).to(device)
    dist_edge_weight = torch.tensor(knn_graph.data, dtype=torch.float).to(device)

    sim_edges = set(zip(sim_edge_index[0].tolist(), sim_edge_index[1].tolist()))
    dist_edges = set(zip(dist_edge_index[0].tolist(), dist_edge_index[1].tolist()))
    common_edges = sim_edges.intersection(dist_edges)

    common_edge_index = torch.tensor(list(zip(*common_edges)), dtype=torch.long).to(device)
    common_edge_weight = torch.ones(common_edge_index.shape[1], dtype=torch.float).to(device)

    x_RNA = torch.tensor(RNA_expression, dtype=torch.float).to(device)
    x_ADT = torch.tensor(ADT_expression, dtype=torch.float).to(device)

    return DualGraphData(
        x_RNA=x_RNA,
        x_ADT=x_ADT,
        sim_edge_index=sim_edge_index,
        sim_edge_weight=sim_edge_weight,
        dist_edge_index=dist_edge_index,
        dist_edge_weight=dist_edge_weight,
        common_edge_index=common_edge_index,
        common_edge_weight=common_edge_weight
    )



def preprocess_universal(adata_RNA, adata_omics2, dataset_name):
    """
    Generic preprocessing for RNA + second modality (ADT/Protein or ATAC).
    """
    # Preprocess RNA (same for both modalities)
    sc.pp.filter_genes(adata_RNA, min_cells=10)
    sc.pp.highly_variable_genes(adata_RNA, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(adata_RNA, target_sum=1e4)
    sc.pp.log1p(adata_RNA)
    sc.pp.scale(adata_RNA)

    RNA_expression = adata_RNA[:, adata_RNA.var['highly_variable']].X
    if scipy.sparse.issparse(RNA_expression):
        RNA_expression = RNA_expression.toarray()

    # Preprocess second modality
    if dataset_name.startswith("10x"):
        # ---------- Protein ----------
        adata_omics2 = adata_omics2[adata_RNA.obs_names].copy()
        adata_omics2 = Protein(adata_omics2)
        ADT_expression = adata_omics2.X
    else:
        # ---------- ATAC ----------
        adata_omics2 = adata_omics2[adata_RNA.obs_names].copy()
        adata_omics2.X = tfidf(adata_omics2.X)
        sc.pp.normalize_per_cell(adata_omics2, counts_per_cell_after=1e4)
        sc.pp.log1p(adata_omics2)
        n_comps = min(60, adata_omics2.shape[1])
        adata_omics2.obsm['feat'] = pca(adata_omics2, n_comps=n_comps)
        ADT_expression = adata_omics2.obsm['feat']

    if scipy.sparse.issparse(ADT_expression):
        ADT_expression = ADT_expression.toarray()

    return RNA_expression, ADT_expression


def preprocess_HLN(adata_RNA, adata_ADT):
    return preprocess_universal(adata_RNA, adata_ADT, "10x_human_lymph_node_A1")



# ------------------------ Model ---------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class DualGCN(nn.Module):
    """
    A dual-stream Graph Convolutional Network for multimodal representation learning.

    Args:
        in_channels (int): Number of input features for the RNA modality.
        hidden_channels (int): Number of hidden units in GCN layers.
        out_channels (int): Number of output embedding dimensions.
        q (int): Dimension of ADT modality.
        dropout (float): Dropout rate.
    """

    def __init__(self, in_channels, hidden_channels, out_channels, q, dropout=0.5):
        super(DualGCN, self).__init__()

        # RNA stream: similarity-based and distance-based GCN branches
        self.x_RNA1 = GCNConv(in_channels, hidden_channels)
        self.x_RNA2 = GCNConv(in_channels, hidden_channels)

        # ADT stream: initial embedding
        self.protein3 = GCNConv(q, out_channels)

        # Project RNA branches to embedding space
        self.sim_conv = GCNConv(hidden_channels, out_channels)
        self.dist_conv = GCNConv(hidden_channels, out_channels)

        # Optional projection from RNA to ADT space
        self.protein = GCNConv(hidden_channels, out_channels)

        # Fusion layers
        self.fusion_layer1 = nn.Sequential(nn.Linear(2 * out_channels, out_channels))
        self.fusion_layer2 = nn.Sequential(nn.Linear(2 * out_channels, out_channels))

        # Decoder layers for reconstruction
        self.dropout = dropout
        self.deconv1 = nn.Linear(out_channels, hidden_channels)
        self.deconv2 = nn.Linear(hidden_channels, in_channels)
        self.deconv4 = nn.Linear(hidden_channels, q)
        self.deconv5 = nn.Linear(hidden_channels, q + in_channels)

    def forward(self, x_RNA, x_ADT, sim_edge_index, sim_edge_weight,
                dist_edge_index, dist_edge_weight, common_edge_index, common_edge_weight):
        """
        Forward pass for dual GCN.
        Returns multiple embeddings: sim_z, dist_z, fused_z, fused_with_adt_z, adt_z
        """
        xs = F.relu(self.x_RNA1(x_RNA, sim_edge_index, sim_edge_weight))
        xs = F.dropout(xs, self.dropout, training=self.training)

        xd = F.relu(self.x_RNA2(x_RNA, dist_edge_index, dist_edge_weight))
        xd = F.dropout(xd, self.dropout, training=self.training)

        x_sim = self.sim_conv(xs, sim_edge_index, sim_edge_weight)
        x_dist = self.dist_conv(xd, dist_edge_index, dist_edge_weight)
        pro = self.protein3(x_ADT, common_edge_index, common_edge_weight)

        combined = torch.cat([x_sim, x_dist], dim=1)
        fused = self.fusion_layer1(combined)

        combined_protein = torch.cat([fused, pro], dim=1)
        fused_pro = self.fusion_layer2(combined_protein)

        return x_sim, x_dist, fused, fused_pro, pro

    # RNA reconstruction
    def reconstruct(self, z):
        x_recon = F.relu(self.deconv1(z))
        return self.deconv2(x_recon)

    # ADT reconstruction
    def reconstruct2(self, z):
        x_recon = F.relu(self.deconv1(z))
        return self.deconv4(x_recon)

    # Joint RNA+ADT reconstruction
    def reconstruct3(self, z):
        x_recon = F.relu(self.deconv1(z))
        return self.deconv5(x_recon)

class Dual(nn.Module):
    """
    Dual Spatially-regularized Deep Multimodal Clustering with Consistency (Dual).

    Args:
        in_channels (int): Dimension of RNA input features.
        hidden_channels (int): Hidden layer dimension in GCN.
        out_channels (int): Output embedding size.
        q (int): Number of ADT features.
        num_clusters (int): Number of target clusters.
        beta (float): Weight for reconstruction loss.
        gamma (float): [Not used here] placeholder for other potential losses.
        delta (float): [Not used here] placeholder for additional losses.
        dropout (float): Dropout rate.
        l1_lambda (float): L1 regularization weight.
        l2_lambda (float): L2 regularization weight.
    """

    def __init__(self, in_channels, hidden_channels, out_channels, q, num_clusters,
                 beta, gamma, delta, dropout,
                 l1_lambda=1e-4, l2_lambda=1e-3):
        super(Dual, self).__init__()
        self.gcn = DualGCN(in_channels, hidden_channels, out_channels, q, dropout)
        self.cluster_layer = nn.Parameter(torch.Tensor(num_clusters, out_channels))
        self.num_clusters = num_clusters
        self.beta = beta
        self.gamma = gamma
        self.delta = delta
        self.l1_lambda = l1_lambda
        self.l2_lambda = l2_lambda
        self.init_weights()

    def init_weights(self):
        nn.init.xavier_uniform_(self.cluster_layer.data)

    def forward(self, x_RNA, x_ADT, sim_edge_index, sim_edge_weight,
                dist_edge_index, dist_edge_weight, common_edge_index, common_edge_weight):
        return self.gcn(x_RNA, x_ADT, sim_edge_index, sim_edge_weight,
                        dist_edge_index, dist_edge_weight, common_edge_index, common_edge_weight)

    def target_distribution(self, q):
        """
        Soft clustering target distribution for deep clustering loss.
        """
        weight = q ** 2 / q.sum(0)
        return (weight.t() / weight.sum(1)).t()

    def compute_regularization_loss(self):
        l1_loss = sum(torch.sum(torch.abs(p)) for p in self.parameters())
        l2_loss = sum(torch.sum(p ** 2) for p in self.parameters())
        return self.l1_lambda * l1_loss + self.l2_lambda * l2_loss

    def compute_losses(self, x_RNA, x_ADT, sim_z, dist_z, fused_z, fused_pro, combined_raw, pro):
        """
        Compute total loss including reconstruction and spatial regularization.
        """
        l_rec = F.mse_loss(combined_raw, self.gcn.reconstruct3(fused_pro))
        l_sim = F.mse_loss(x_RNA, self.gcn.reconstruct(sim_z))
        l_dist = F.mse_loss(x_RNA, self.gcn.reconstruct(dist_z))
        l_adt = F.mse_loss(x_ADT, self.gcn.reconstruct2(pro))

        l_spatial = self.spatial_regularization_loss(fused_z, dist_edge_index=self.gcn_input.dist_edge_index,
                                                     dist_edge_weight=self.gcn_input.dist_edge_weight)

        reg_loss = self.compute_regularization_loss()
        total_loss = self.beta * (l_rec + l_sim + l_dist + l_adt) + self.gamma * l_spatial + self.delta * reg_loss

        return total_loss, l_rec

    def cosine_similarity(self, emb):
        mat = torch.matmul(emb, emb.T)
        norm = torch.norm(emb, p=2, dim=1).reshape((emb.shape[0], 1))
        mat = torch.div(mat, torch.matmul(norm, norm.T))
        mat = torch.where(torch.isnan(mat), torch.zeros_like(mat), mat)
        mat = mat - torch.diag_embed(torch.diag(mat))
        return mat

    def spatial_regularization_loss(self, emb, dist_edge_index, dist_edge_weight):
        """
        Spatial regularization to encourage local smoothness and contrastive separation.
        """
        num_nodes = emb.size(0)
        graph_nei = torch.sparse_coo_tensor(dist_edge_index, torch.ones_like(dist_edge_weight),
                                            size=(num_nodes, num_nodes)).to_dense()
        graph_neg = 1 - graph_nei
        sim_mat = torch.sigmoid(self.cosine_similarity(emb))

        neigh_loss = torch.mul(graph_nei, torch.log(sim_mat + 1e-10)).mean()
        neg_loss = torch.mul(graph_neg, torch.log(1 - sim_mat + 1e-10)).mean()

        return -(neigh_loss + neg_loss) / 2

class TriGCN(nn.Module):
    """
    A triple-stream Graph Convolutional Network for multimodal representation learning (3 Modalities).

    Args:
        in_channels (int): Number of input features for the RNA modality.
        hidden_channels (int): Number of hidden units in GCN layers.
        out_channels (int): Number of output embedding dimensions.
        q (int): Dimension of ADT (Protein) modality.
        dropout (float): Dropout rate.
    """

    def __init__(self, in_channels, hidden_channels, out_channels, q, dropout=0.5):
        super(TriGCN, self).__init__()

        # RNA stream: similarity-based and distance-based GCN branches
        self.x_RNA1 = GCNConv(in_channels, hidden_channels)
        self.x_RNA2 = GCNConv(in_channels, hidden_channels)

        # ADT stream: initial embedding
        self.protein3 = GCNConv(q, out_channels)

        # Project RNA branches to embedding space
        self.sim_conv = GCNConv(hidden_channels, out_channels)
        self.dist_conv = GCNConv(hidden_channels, out_channels)

        # Fusion layers
        # Fusion 1: Combine RNA Similarity and Distance features
        self.fusion_layer1 = nn.Sequential(nn.Linear(2 * out_channels, out_channels))
        # Fusion 2: Combine Fused RNA features with ADT features
        self.fusion_layer2 = nn.Sequential(nn.Linear(2 * out_channels, out_channels))

        # Decoder layers for reconstruction
        self.dropout = dropout
        self.deconv1 = nn.Linear(out_channels, hidden_channels)

        self.deconv2 = nn.Linear(hidden_channels, in_channels)       # For RNA recon
        self.deconv4 = nn.Linear(hidden_channels, q)                 # For ADT recon
        self.deconv5 = nn.Linear(hidden_channels, q + in_channels)   # For Combined recon

    def forward(self, x_RNA, x_ADT, sim_edge_index, sim_edge_weight,
                dist_edge_index, dist_edge_weight, common_edge_index, common_edge_weight):
        """
        Forward pass for dual GCN.
        Returns multiple embeddings: x_sim, x_dist, fused (RNA), fused_pro (RNA+ADT), pro
        """
        # RNA Encoder
        xs = F.relu(self.x_RNA1(x_RNA, sim_edge_index, sim_edge_weight))
        xs = F.dropout(xs, self.dropout, training=self.training)

        xd = F.relu(self.x_RNA2(x_RNA, dist_edge_index, dist_edge_weight))
        xd = F.dropout(xd, self.dropout, training=self.training)

        x_sim = self.sim_conv(xs, sim_edge_index, sim_edge_weight)
        x_dist = self.dist_conv(xd, dist_edge_index, dist_edge_weight)

        # ADT Encoder
        pro = self.protein3(x_ADT, common_edge_index, common_edge_weight)

        # Fusion
        combined = torch.cat([x_sim, x_dist], dim=1)
        fused = self.fusion_layer1(combined)

        combined_protein = torch.cat([fused, pro], dim=1)
        fused_pro = self.fusion_layer2(combined_protein)

        return x_sim, x_dist, fused, fused_pro, pro

    # RNA reconstruction
    def reconstruct(self, z):
        x_recon = F.relu(self.deconv1(z))
        return self.deconv2(x_recon)

    # ADT reconstruction
    def reconstruct2(self, z):
        x_recon = F.relu(self.deconv1(z))
        return self.deconv4(x_recon)

    # Joint RNA+ADT reconstruction
    def reconstruct3(self, z):
        x_recon = F.relu(self.deconv1(z))
        return self.deconv5(x_recon)


class Tri(nn.Module):
    """
    Triple Spatially-regularized Deep Multimodal Clustering with Consistency (Tri)
    for 3 Modalities.

    Args:
        in_channels (int): Dimension of RNA input features.
        hidden_channels (int): Hidden layer dimension in GCN.
        out_channels (int): Output embedding size.
        q (int): Number of ADT features.
        num_clusters (int): Number of target clusters.
        beta (float): Weight for reconstruction loss.
        gamma (float): Weight for spatial regularization loss.
        delta (float): Weight for parameter regularization loss.
        dropout (float): Dropout rate.
        l1_lambda (float): L1 regularization weight.
        l2_lambda (float): L2 regularization weight.
    """

    def __init__(self, in_channels, hidden_channels, out_channels, q, num_clusters,
                 beta, gamma, delta, dropout,
                 l1_lambda=1e-5, l2_lambda=1e-4):
        super(Tri, self).__init__()
        self.gcn = TriGCN(in_channels, hidden_channels, out_channels, q, dropout=dropout)
        self.cluster_layer = nn.Parameter(torch.Tensor(num_clusters, out_channels))
        self.num_clusters = num_clusters

        self.beta = beta
        self.gamma = gamma
        self.delta = delta
        self.l1_lambda = l1_lambda
        self.l2_lambda = l2_lambda

        self.init_weights()

    def init_weights(self):
        nn.init.xavier_uniform_(self.cluster_layer.data)

    def forward(self, x_RNA, x_ADT, sim_edge_index, sim_edge_weight,
                dist_edge_index, dist_edge_weight, common_edge_index, common_edge_weight):
        return self.gcn(x_RNA, x_ADT, sim_edge_index, sim_edge_weight,
                        dist_edge_index, dist_edge_weight, common_edge_index, common_edge_weight)

    def target_distribution(self, q):
        """
        Soft clustering target distribution for deep clustering loss.
        """
        weight = q ** 2 / q.sum(0)
        return (weight.t() / weight.sum(1)).t()

    def compute_regularization_loss(self):
        l1_loss = 0
        l2_loss = 0
        for param in self.parameters():
            l1_loss += torch.sum(torch.abs(param))
            l2_loss += torch.sum(param.pow(2))
        return self.l1_lambda * l1_loss + self.l2_lambda * l2_loss

    def compute_losses(self, x_RNA, x_ADT, sim_z, dist_z, fused_z, fused_pro, combined_raw, pro, dist_edge_index, dist_edge_weight):
        """
        Compute total loss including reconstruction and spatial regularization.
        """
        # Reconstruction losses
        l_rec = F.mse_loss(combined_raw, self.gcn.reconstruct3(fused_pro))
        l_sim = F.mse_loss(x_RNA, self.gcn.reconstruct(sim_z))
        l_dist = F.mse_loss(x_RNA, self.gcn.reconstruct(dist_z))
        l_adt = F.mse_loss(x_ADT, self.gcn.reconstruct2(pro))

        # Spatial regularization loss (applied to RNA-fused latent space or final fused space)
        l_spatial = self.spatial_regularization_loss(fused_z, dist_edge_index, dist_edge_weight)

        # L1/L2 Parameter Regularization
        reg_loss = self.compute_regularization_loss()

        # Total loss calculation
        total_loss = self.beta * (l_rec + l_sim + l_dist + l_adt) + self.gamma * l_spatial + self.delta * reg_loss

        return total_loss, l_rec

    def cosine_similarity(self, emb):
        # Cosine similarity matrix computation
        emb_norm = F.normalize(emb, p=2, dim=1, eps=1e-8)
        mat = torch.matmul(emb_norm, emb_norm.T)
        mat = mat - torch.diag_embed(torch.diag(mat))  # Remove self-similarity (diagonal)
        return mat

    def spatial_regularization_loss(self, emb, dist_edge_index, dist_edge_weight):
        """
        Spatial regularization to encourage local smoothness and contrastive separation.
        """
        num_nodes = emb.size(0)
        # Create neighborhood adjacency matrix
        graph_nei = torch.sparse_coo_tensor(
            dist_edge_index,
            torch.ones_like(dist_edge_weight),
            size=(num_nodes, num_nodes)
        ).to_dense()
        graph_neg = 1 - graph_nei

        # Sigmoid scaled similarity
        sim_mat = torch.sigmoid(self.cosine_similarity(emb))

        # Attract neighbors, repel non-neighbors
        neigh_loss = torch.mul(graph_nei, torch.log(sim_mat + 1e-10)).mean()
        neg_loss = torch.mul(graph_neg, torch.log(1 - sim_mat + 1e-10)).mean()

        return -(neigh_loss + neg_loss) / 2


# ------------------------ Train ---------------------------------------

import torch
import torch.nn.functional as F
from sklearn.cluster import KMeans
import numpy as np
# from model import Dual
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score
)


def evaluate_model(model, data):
    """
    Run the model in evaluation mode and return fused predictions and embeddings.

    Args:
        model (Dual): The trained model.
        data (DualGraphData): Graph data with features and edges.

    Returns:
            - fused_pro: fused probability (output of final prediction layer),
            - sim_z: RNA similarity graph embedding,
            - dist_z: spatial distance graph embedding.
    """
    model.eval()
    with torch.no_grad():
        sim_z, dist_z, fused_z ,fused_pro,pro = model(data.x_RNA, data.x_ADT, data.sim_edge_index, data.sim_edge_weight ,data.dist_edge_index, data.dist_edge_weight, data.common_edge_index , data.common_edge_weight)
    return fused_pro.cpu().numpy(), sim_z.cpu().numpy(), dist_z.cpu().numpy()

def cluster_embeddings(embeddings, num_clusters):
    """
    Perform KMeans clustering on the embeddings.

    Args:
        embeddings (np.ndarray): Feature embeddings.
        num_clusters (int): Number of clusters.

    Returns:
         Predicted cluster labels.
    """
    kmeans = KMeans(n_clusters=num_clusters, n_init=10, random_state=42)
    return kmeans.fit_predict(embeddings)

def evaluate_model_performance(predicted_labels, true_labels):
    ari = adjusted_rand_score(true_labels, predicted_labels)
    nmi = normalized_mutual_info_score(true_labels, predicted_labels)
    return ari, nmi


def initialize_model(data, input_dim, adt_dim, args):
    """
       Initialize the Dual model with hyperparameters.

       Args:
           data (DualGraphData): Input graph data.
           input_dim (int): Input dimension of RNA features.
           adt_dim (int): Input dimension of ADT features.
           args (Namespace): Hyperparameters and settings.

       Returns:
           Dual: Initialized model instance.
       """
    model = Dual(
        in_channels=input_dim,
        hidden_channels=args.hidden_dim,
        out_channels=args.out_dim,
        q=adt_dim,
        num_clusters=args.num_clusters,
        beta=args.beta,
        gamma=args.gamma,
        delta=args.delta,
        dropout=args.dropout
    ).to(args.device)

    return model


def train_model(model, data, args ,true_labels):
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    model.train()

    combined_raw = torch.cat([data.x_RNA, data.x_ADT], dim=1)
    best_sil = -1
    best_embeddings = None
    best_labels = None

    for epoch in range(args.epochs):
        optimizer.zero_grad()

        sim_z, dist_z, fused_z, fused_pro, pro = model(
            data.x_RNA, data.x_ADT,
            data.sim_edge_index, data.sim_edge_weight,
            data.dist_edge_index, data.dist_edge_weight,
            data.common_edge_index, data.common_edge_weight
        )


        model.gcn_input = data  # for spatial regularization

        loss, l_rec = model.compute_losses(
            data.x_RNA, data.x_ADT,
            sim_z, dist_z, fused_z, fused_pro, combined_raw, pro
        )

        loss.backward()
        optimizer.step()

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1}, Total Loss: {loss.item():.4f}, Recon Loss: {l_rec.item():.4f}")

        # Evaluate clustering performance
        embeddings, _, _ = evaluate_model(model, data)

        predicted_labels = cluster_embeddings(
            embeddings,
            args.num_clusters
        )

        sil = silhouette_score(
            embeddings,
            predicted_labels
        )

        if sil > best_sil:
            best_sil = sil
            best_embeddings = embeddings.copy()
            best_labels = predicted_labels.copy()

        print(f"Epoch {epoch+1:3d} | Silhouette: {sil:.4f}")

    ari, nmi = evaluate_model_performance(
        best_labels,
        true_labels
    )

    print("\n==============================")
    print("Best checkpoint selected by Silhouette")
    print(f"Best Silhouette : {best_sil:.4f}")
    print(f"ARI             : {ari:.4f}")
    print(f"NMI             : {nmi:.4f}")
    print("==============================")

    return model, best_embeddings, best_labels



# ------------------------ Argument pass and random seed  ---------------------------------------

import argparse
import os
import random
import numpy as np
import torch
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    v_measure_score,
    silhouette_score
)

def set_seed(seed=2024):
    """
    Set random seed for reproducibility across Python, NumPy, PyTorch, and CUDA.

    Parameters
    ----------
    seed : int, default=2024
        Random seed value.

    Returns
    -------
    None
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

# Set parameters.
parser = argparse.ArgumentParser()
parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu')
parser.add_argument('--hidden_dim', type=int, default=512)
parser.add_argument('--out_dim', type=int, default=64)
parser.add_argument('--num_clusters', type=int, default=10)
parser.add_argument('--beta', type=float, default=25)
parser.add_argument('--gamma', type=float, default=10)
parser.add_argument('--delta', type=float, default=1)
parser.add_argument('--dropout', type=float, default=0)
parser.add_argument('--lr', type=float, default=1e-3)
parser.add_argument('--epochs', type=int, default=350)

# Set device to GPU if available, otherwise use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
args, unknown = parser.parse_known_args()
args.device = device

# -------------------------------------- Run Configuration ---------------------------------------

choices = [
    ("10x_human_lymph_node_A1",
     "https://drive.google.com/drive/folders/10z1N4MwW8Y49o8GlkYGBKVx1N7fiMuyC"),
    ("10x_human_lymph_node_D1",
     "https://drive.google.com/drive/folders/1-g_Ca2XMaMXF-MisuVY-wobWDX86O6zz"),
    ("Mouse_Brain_E11_S1",
     "https://drive.google.com/drive/folders/1zRwDJrYnks0LRzlAVRqPU7jE_OcStgPo"),
    ("Mouse_Brain_E13_S1",
     "https://drive.google.com/drive/folders/1GOufwIRjjfcd9Bi2GKtebzKoPCg2jVud"),
    ("Mouse_Brain_E15_S1",
     "https://drive.google.com/drive/folders/1rHkTL5OF5qPsEERypRGMS51SjUQ69tdD"),
    ("Mouse_Brain_E18_S1",
     "https://drive.google.com/drive/folders/1Xj1LNIAY93biS6JIMKNRODn5GvtCKADB"),
]

# Select which datasets to run (indices 0 to 5)
DATASET_INDICES = [0, 1, 2, 3, 4, 5]

# Define seeds to loop over
SEEDS = [
    13,
    2560,
    641,
    1892,
    1173,
    69,
    2024,
    231,
    1971,
    2497,
    338,
    3127,
    2001,
    2022,
    574,
    2428,
    999,
    1187,
    42,
    3999
]

all_results = []

for dataset_idx in DATASET_INDICES:
    dataset_name, folder_url = choices[dataset_idx]

    print("\n" + "#"*80)
    print(f" STARTING DATASET: {dataset_name} ".center(80, "#"))
    print("#"*80)

    # ------------------ Download & Path Configuration ------------------
    base = f"data/{dataset_name}"
    os.makedirs(base, exist_ok=True)

    rna_path = os.path.join(base, "adata_RNA.h5ad")

    # Human datasets have ADT; Mouse datasets have ATAC
    if dataset_name.startswith("10x"):
        other_path = os.path.join(base, "adata_ADT.h5ad")
        annotation_path = os.path.join(base, "annotation.csv")
        gt_column = "manual-anno"
        data_type = '10x'
    else:
        other_path = os.path.join(base, "adata_ATAC.h5ad")
        annotation_path = os.path.join(base, "anno.csv")
        gt_column = "cluster"
        data_type = 'Spatial-epigenome-transcriptome'

    # Check if files already exist locally before triggering download
    if not os.path.exists(rna_path) or not os.path.exists(other_path) or not os.path.exists(annotation_path):
        print(f"Downloading dataset files into: {base}")
        os.system(f'gdown --folder "{folder_url}" --output "{base}"')
    else:
        print(f"Dataset files already exist at {base}. Skipping download.")

    # ------------------ Load AnnData & Annotations ------------------
    adata_omics1 = sc.read_h5ad(rna_path)
    adata_omics2 = sc.read_h5ad(other_path)

    adata_omics1.var_names_make_unique()
    adata_omics2.var_names_make_unique()

    anno_df = pd.read_csv(annotation_path, index_col=0)
    adata_omics1.obs['ground_truth'] = anno_df[gt_column]
    adata_omics2.obs['ground_truth'] = anno_df[gt_column]

    # Preprocess RNA and second modality data (normalization, filtering, etc.)
    RNA_data, ADT_data = preprocess_universal(adata_omics1, adata_omics2, dataset_name)

    cell_positions = adata_omics1.obsm['spatial']
    graph_data = build_dual_graph(RNA_data, ADT_data, cell_positions, device=device)

    # ------------------ Seed Iterations ------------------
    dataset_results = []
    n_ground_truth = adata_omics1.obs["ground_truth"].nunique()
    print(f"Number of ground truth classes: {n_ground_truth}")
    args.num_clusters = n_ground_truth

    for seed in SEEDS:
        print(f"\n" + "-"*60)
        print(f" Dataset: {dataset_name} | Seed: {seed} ".center(60, "-"))
        print("-"*60)

        # 1. Set seed
        set_seed(seed)

        # 2. Model Init
        model = initialize_model(graph_data, RNA_data.shape[1], ADT_data.shape[1], args)

        # 3. Model Training
        model, final_embeddings, final_labels = train_model(model, graph_data, args, adata_omics1.obs['ground_truth'])

        # 4. Score Generating
        y_true = adata_omics1.obs['ground_truth'].astype(str)
        y_pred = final_labels.astype(str)

        ari = adjusted_rand_score(y_true, y_pred)
        nmi = normalized_mutual_info_score(y_true, y_pred)
        ami = adjusted_mutual_info_score(y_true, y_pred)
        homogeneity = homogeneity_score(y_true, y_pred)
        v_measure = v_measure_score(y_true, y_pred)

        # Calculate Silhouette Score using final_embeddings and final_labels
        sil_score = silhouette_score(final_embeddings, final_labels)

        print(f"\nResult for dataset: {dataset_name} | seed: {seed}")
        print(f"ARI: {ari:.4f} | NMI: {nmi:.4f} | Silhouette: {sil_score:.4f}")

        # Store seed results
        res_dict = {
            'dataset': dataset_name,
            'seed': seed,
            'ARI': ari,
            'NMI': nmi,
            'AMI': ami,
            'Homogeneity': homogeneity,
            'V-measure': v_measure,
            'Silhouette': sil_score
        }
        dataset_results.append(res_dict)
        all_results.append(res_dict)

        # ------------------ Plotting & Visualization ------------------
        adata = adata_omics1.copy()
        adata.obsm['ARISE'] = final_embeddings
        adata.obs['ARISE'] = final_labels.astype(str)
        if 'anno' not in adata.obs.columns and 'ground_truth' in adata.obs.columns:
            adata.obs['anno'] = adata.obs['ground_truth']

        fig, ax_list = plt.subplots(1, 3, figsize=(11, 3))
        sc.pp.neighbors(adata, use_rep='ARISE', n_neighbors=10)
        sc.tl.umap(adata)

        sc.pl.umap(adata, color='ARISE', ax=ax_list[0], title='ARISE', s=20, show=False)
        sc.pl.embedding(adata, basis='spatial', color='ARISE', ax=ax_list[1], title='ARISE', s=20, show=False)
        sc.pl.embedding(adata, basis='spatial', color='anno', ax=ax_list[2], title='anno', s=20, show=False)
        plt.tight_layout(w_pad=0.3)

        os.makedirs(f"plots/{dataset_name}", exist_ok=True)
        plot_path = f"plots/{dataset_name}/umap_seed_{seed}.png"
        plt.savefig(plot_path, dpi=150)
        plt.close(fig)

    # Write dataset summary statistics and save csv for this dataset
    df_ds = pd.DataFrame(dataset_results)
    print("\n" + "="*80)
    print(f" RESULTS SUMMARY FOR {dataset_name} ".center(80, "="))
    print("="*80)
    print(df_ds.to_string(index=False))
    print("-"*80)
    print("Summary Statistics:")
    print(df_ds.describe().loc[['mean', 'std']])
    print("="*80)

    # Generate text summary with Mean ± SD format for this dataset
    summary_lines = [f"Summary Stats for {dataset_name}:"]
    metrics_list = ['ARI', 'NMI', 'AMI', 'Homogeneity', 'V-measure', 'Silhouette']
    for metric in metrics_list:
        mean_val = df_ds[metric].mean()
        std_val = df_ds[metric].std()
        summary_lines.append(f"{metric} = {mean_val:.4f} ± {std_val:.4f}")

    dataset_summary_str = "\n".join(summary_lines)
    print(dataset_summary_str)
    print("="*80)

    os.makedirs("results", exist_ok=True)

    # Save the text summary for this dataset
    with open(f"results/summary_{dataset_name}.txt", "w") as f:
        f.write(dataset_summary_str)

    df_ds.to_csv(f"results/metrics_{dataset_name}.csv", index=False)
    print(f"Saved dataset metrics to 'results/metrics_{dataset_name}.csv'")
    print(f"Saved dataset summary to 'results/summary_{dataset_name}.txt'")

    # Generate publication-quality boxplot + stripplot for this dataset
    plt.figure(figsize=(10, 6))
    df_ds_long = df_ds.melt(
        id_vars=['seed'],
        value_vars=metrics_list,
        var_name='Metric',
        value_name='Score'
    )
    sns.set_theme(style="whitegrid")
    ax = sns.boxplot(
        x='Metric',
        y='Score',
        data=df_ds_long,
        palette='Set2',
        showfliers=False,
        width=0.5
    )
    sns.stripplot(
        x='Metric',
        y='Score',
        data=df_ds_long,
        color='black',
        alpha=0.6,
        jitter=0.2,
        size=5,
        ax=ax
    )
    plt.title(f"Variability of Metrics across 20 Seeds ({dataset_name})", fontsize=14, pad=15)
    plt.xlabel("Metric", fontsize=12, labelpad=10)
    plt.ylabel("Score", fontsize=12, labelpad=10)
    plt.ylim(0, 1.05)
    plt.tight_layout()

    boxplot_path = f"results/boxplot_{dataset_name}.png"
    plt.savefig(boxplot_path, dpi=300)
    plt.close()
    print(f"Saved variability boxplot to '{boxplot_path}'\n")

# -------------------------------------- Global Summary statistics ---------------------------------------

df_all = pd.DataFrame(all_results)
os.makedirs("results", exist_ok=True)
df_all.to_csv("results/metrics_all_datasets.csv", index=False)

# Compile global text summary
global_summary_lines = [
    "="*80,
    "GLOBAL SUMMARY: MEAN ± STD ACROSS 20 SEEDS",
    "="*80
]

for d_name in df_all['dataset'].unique():
    df_d = df_all[df_all['dataset'] == d_name]
    global_summary_lines.append(f"\nDataset: {d_name}")
    global_summary_lines.append("-" * len(f"Dataset: {d_name}"))
    for metric in ['ARI', 'NMI', 'AMI', 'Homogeneity', 'V-measure', 'Silhouette']:
        mean_val = df_d[metric].mean()
        std_val = df_d[metric].std()
        global_summary_lines.append(f"{metric} = {mean_val:.4f} ± {std_val:.4f}")
global_summary_lines.append("\n" + "="*80)

global_summary_str = "\n".join(global_summary_lines)
print(global_summary_str)

summary_stats_path = "results/summary_statistics.txt"
with open(summary_stats_path, "w") as f:
    f.write(global_summary_str)

print("\n" + "#"*80)
print(" GLOBAL RESULTS ACROSS ALL DATASETS ".center(80, "#"))
print("#"*80)
print(df_all.groupby('dataset').mean())
print("#"*80)
print(f"Saved combined metrics to 'results/metrics_all_datasets.csv'")
print(f"Saved global summary statistics to '{summary_stats_path}'")

# Generate comparative global boxplot across all datasets
df_all_long = df_all.melt(
    id_vars=['dataset', 'seed'],
    value_vars=['ARI', 'NMI', 'AMI', 'Homogeneity', 'V-measure', 'Silhouette'],
    var_name='Metric',
    value_name='Score'
)

fig, axes = plt.subplots(2, 3, figsize=(18, 12), sharey=True)
axes = axes.flatten()
metrics_list = ['ARI', 'NMI', 'AMI', 'Homogeneity', 'V-measure', 'Silhouette']

sns.set_theme(style="whitegrid")
for i, metric in enumerate(metrics_list):
    df_metric = df_all_long[df_all_long['Metric'] == metric]
    sns.boxplot(
        ax=axes[i],
        x='dataset',
        y='Score',
        data=df_metric,
        palette='Set3',
        showfliers=False
    )
    sns.stripplot(
        ax=axes[i],
        x='dataset',
        y='Score',
        data=df_metric,
        color='black',
        alpha=0.5,
        jitter=0.15,
        size=4
    )
    axes[i].set_title(f"{metric} Comparison", fontsize=14)
    axes[i].set_xlabel("Dataset", fontsize=12)
    axes[i].set_ylabel("Score", fontsize=12)
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
    axes[i].set_ylim(0, 1.05)

plt.suptitle("Performance Comparison Across All Datasets & 20 Seeds", fontsize=18, y=0.98)
plt.tight_layout()
global_plot_path = "results/global_boxplot_comparison.png"
plt.savefig(global_plot_path, dpi=300)
plt.close()
print(f"Saved global comparison boxplot to '{global_plot_path}'")

# -------------------------------------- End ---------------------------------------









################################################################################
################## STARTING DATASET: 10x_human_lymph_node_A1 ###################
################################################################################


/tmp/ipykernel_1045/546894004.py:981: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata_omics1 = sc.read_h5ad(rna_path)
/tmp/ipykernel_1045/546894004.py:982: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata_omics2 = sc.read_h5ad(other_path)
/usr/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
/tmp/ipykernel_1045/546894004.py:252: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  dist_edge_index = torch.tensor(knn_graph.nonzero(), dtype=torch.long).to(device)


Number of ground truth classes: 10

------------------------------------------------------------
------- Dataset: 10x_human_lymph_node_A1 | Seed: 13 --------
------------------------------------------------------------


/tmp/ipykernel_1045/546894004.py:488: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  graph_nei = torch.sparse_coo_tensor(dist_edge_index, torch.ones_like(dist_edge_weight),


Streaming output truncated to the last 5000 lines.
Epoch 220, Total Loss: 81.7590, Recon Loss: 0.9391
Epoch 220 | Silhouette: 0.2030
Epoch 221 | Silhouette: 0.1993
Epoch 222 | Silhouette: 0.2063
Epoch 223 | Silhouette: 0.2054
Epoch 224 | Silhouette: 0.2058
Epoch 225 | Silhouette: 0.2048
Epoch 226 | Silhouette: 0.2045
Epoch 227 | Silhouette: 0.2044
Epoch 228 | Silhouette: 0.2040
Epoch 229 | Silhouette: 0.2042
Epoch 230, Total Loss: 81.6697, Recon Loss: 0.9375
Epoch 230 | Silhouette: 0.1976
Epoch 231 | Silhouette: 0.1993
Epoch 232 | Silhouette: 0.1937
Epoch 233 | Silhouette: 0.2038
Epoch 234 | Silhouette: 0.2033
Epoch 235 | Silhouette: 0.1959
Epoch 236 | Silhouette: 0.2017
Epoch 237 | Silhouette: 0.1995
Epoch 238 | Silhouette: 0.2008
Epoch 239 | Silhouette: 0.1888
Epoch 240, Total Loss: 81.7063, Recon Loss: 0.9386
Epoch 240 | Silhouette: 0.2011
Epoch 241 | Silhouette: 0.2002
Epoch 242 | Silhouette: 0.2001
Epoch 243 | Silhouette: 0.1979
Epoch 244 | Silhouette: 0.1835
Epoch 245 | Silhouett

...

Streaming output truncated to the last 5000 lines.
Epoch 220, Total Loss: 89.0504, Recon Loss: 0.8061
Epoch 220 | Silhouette: 0.1169
Epoch 221 | Silhouette: 0.1169
Epoch 222 | Silhouette: 0.1147
Epoch 223 | Silhouette: 0.1224
Epoch 224 | Silhouette: 0.1128
Epoch 225 | Silhouette: 0.1155
Epoch 226 | Silhouette: 0.1140
Epoch 227 | Silhouette: 0.1127
Epoch 228 | Silhouette: 0.1128
Epoch 229 | Silhouette: 0.1129
Epoch 230, Total Loss: 88.7761, Recon Loss: 0.8000
Epoch 230 | Silhouette: 0.1128
Epoch 231 | Silhouette: 0.1142
Epoch 232 | Silhouette: 0.1129
Epoch 233 | Silhouette: 0.1187
Epoch 234 | Silhouette: 0.1139
Epoch 235 | Silhouette: 0.1131
Epoch 236 | Silhouette: 0.1115
Epoch 237 | Silhouette: 0.1131
Epoch 238 | Silhouette: 0.1128
Epoch 239 | Silhouette: 0.1109
Epoch 240, Total Loss: 88.5117, Recon Loss: 0.7940
Epoch 240 | Silhouette: 0.1200
Epoch 241 | Silhouette: 0.1120
Epoch 242 | Silhouette: 0.1087
Epoch 243 | Silhouette: 0.1062
Epoch 244 | Silhouette: 0.1110
Epoch 245 | Silhouett

/tmp/ipykernel_1045/546894004.py:1110: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(


Saved variability boxplot to 'results/boxplot_Mouse_Brain_E18_S1.png'

GLOBAL SUMMARY: MEAN ± STD ACROSS 20 SEEDS

Dataset: 10x_human_lymph_node_A1
--------------------------------
ARI = 0.3266 ± 0.0097
NMI = 0.4087 ± 0.0089
AMI = 0.4045 ± 0.0089
Homogeneity = 0.4125 ± 0.0050
V-measure = 0.4087 ± 0.0089
Silhouette = 0.2617 ± 0.0074

Dataset: 10x_human_lymph_node_D1
--------------------------------
ARI = 0.2632 ± 0.0182
NMI = 0.3586 ± 0.0054
AMI = 0.3532 ± 0.0054
Homogeneity = 0.3938 ± 0.0059
V-measure = 0.3586 ± 0.0054
Silhouette = 0.2143 ± 0.0089

Dataset: Mouse_Brain_E11_S1
---------------------------
ARI = 0.4432 ± 0.0245
NMI = 0.5566 ± 0.0197
AMI = 0.5516 ± 0.0200
Homogeneity = 0.6395 ± 0.0238
V-measure = 0.5566 ± 0.0197
Silhouette = 0.1954 ± 0.0078

Dataset: Mouse_Brain_E13_S1
---------------------------
ARI = 0.2463 ± 0.0166
NMI = 0.4472 ± 0.0140
AMI = 0.4380 ± 0.0143
Homogeneity = 0.5353 ± 0.0141
V-measure = 0.4472 ± 0.0140
Silhouette = 0.1327 ± 0.0042

Dataset: Mouse_Brain_E15_

/tmp/ipykernel_1045/546894004.py:1192: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_1045/546894004.py:1213: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
/tmp/ipykernel_1045/546894004.py:1192: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_1045/546894004.py:1213: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
/tmp/ipykernel_1045/546894004.py:119

Saved global comparison boxplot to 'results/global_boxplot_comparison.png'
